# Notebook de Fine-tuning 

En este notebook se entrena DistilBERT para clasificar mensajes de clientes 
en 77 intenciones posibles. El pipeline cubre desde la preparación de datos 
hasta guardar el modelo listo para ser consumido por el worker.

## 1. Set up inicial

Acá cargamos el dataset ya limpio y reconstruimos todo lo necesario para entrenar: 
- el split estratificado para mantener la proporción de clases
- la tokenización para convertir texto a tensores
- los pesos por clase para no sesgar el modelo hacia las clases más frecuentes
- el mapeo de ids a etiquetas para poder interpretar las predicciones.


In [12]:
%pip install scikit-learn

import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/train_clean.csv')

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
NUM_LABELS = df['label_id'].nunique()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label_id'],
    test_size=0.2, random_state=42, stratify=df['label_id']
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_encodings = tokenizer(X_train.tolist(), truncation=True, padding=True, max_length=MAX_LENGTH)
test_encodings  = tokenizer(X_test.tolist(),  truncation=True, padding=True, max_length=MAX_LENGTH)

classes, counts = np.unique(y_train, return_counts=True)
weights = len(y_train) / (len(classes) * counts)
class_weights = torch.tensor(weights, dtype=torch.float)

label_mapping = df[['label', 'label_id']].drop_duplicates().sort_values('label_id')
id2label = dict(zip(label_mapping['label_id'], label_mapping['label']))

print(f'Clases: {NUM_LABELS} | Train: {len(X_train):,} | Val: {len(X_test):,} | Device: {device}')


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Clases: 77 | Train: 8,256 | Val: 2,064 | Device: cpu


## 2. Dataset de PyTorch

El Trainer de Hugging Face necesita los datos en un formato específico, así que creamos una clase IntentDataset que envuelve los encodings y las etiquetas como tensores listos para entrenar.

In [13]:
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = IntentDataset(train_encodings, y_train.tolist())
val_dataset   = IntentDataset(test_encodings,  y_test.tolist())

print('Train dataset:', len(train_dataset))
print('Val dataset:  ', len(val_dataset))

Train dataset: 8256
Val dataset:   2064


## 3. Modelo base y función de pérdida

Cargamos DistilBERT preentrenado y le agregamos una cabeza de clasificación para nuestras 77 clases. Como el dataset está desbalanceado, reemplazamos la pérdida estándar por una cross-entropy ponderada que le da más peso a las clases minoritarias.

In [14]:
try:
    from transformers import (
        AutoModelForSequenceClassification,
        TrainingArguments, Trainer,
        EarlyStoppingCallback,
    )
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'accelerate'])
    from transformers import (
        AutoModelForSequenceClassification,
        TrainingArguments, Trainer,
        EarlyStoppingCallback,
    )

import torch.nn.functional as F
from sklearn.metrics import f1_score

class_weights_device = class_weights.to(device)

class WeightedTrainer(Trainer):
    """Aplica class weights en cross-entropy para clases desbalanceadas."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        loss    = F.cross_entropy(outputs.logits, labels, weight=class_weights_device)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'f1_weighted': round(f1_score(labels, preds, average='weighted', zero_division=0), 4),
        'f1_macro':    round(f1_score(labels, preds, average='macro',    zero_division=0), 4),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(device)
print(f'Modelo cargado | Parámetros: {sum(p.numel() for p in model.parameters())//1_000_000}M')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3730.99it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado | Parámetros: 67M


## 4. Fine-tuning

Entrenamos 5 épocas con early stopping para no sobreentrenar. El modelo guarda el mejor checkpoint según F1 weighted en validación.

In [15]:
%pip install 'accelerate>=1.1.0' -q


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'accelerate": Expected package name at the start of dependency specifier
    'accelerate
    ^


In [ ]:

import os
MODEL_DIR = '../model'
os.makedirs(MODEL_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir='../checkpoints',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=100,          
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

C:\Users\males\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## 5. Evaluacion

En base a las predicciones generadas calculamos el F1 score y el F1 macro. El modelo alcanzó F1 weighted de 0.85 y F1 macro de 0.86 en validación. Usamos F1 en lugar de accuracy porque con 77 clases desbalanceadas, no solo porque será la métrica de evaluación, también porque para clases desbalanceadas accuracy puede ser engañosa.


In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = y_test.values

print('=' * 60)
print(f"F1 weighted : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1 macro    : {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
print('=' * 60)
print(classification_report(
    y_true, y_pred,
    target_names=list(id2label.values()),
    zero_division=0,
))

C:\Users\males\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


F1 weighted : 0.8539
F1 macro    : 0.8557
                                                  precision    recall  f1-score   support

              transfer_not_received_by_recipient       0.93      0.43      0.59        30
                         pending_cash_withdrawal       0.92      0.89      0.91        27
                        declined_cash_withdrawal       0.83      0.86      0.85        29
         balance_not_updated_after_bank_transfer       0.73      0.85      0.79        39
                               declined_transfer       0.92      0.74      0.82        31
                                   top_up_limits       0.80      0.92      0.86        26
                        card_payment_fee_charged       0.80      1.00      0.89        32
                              passcode_forgotten       0.95      1.00      0.98        21
                           declined_card_payment       0.69      0.85      0.76        26
                                  request_refund       0.

## 6. Guardar modelo

Guardamos modelo, tokenizer y mapeo de etiquetas juntos en ../model/ para que la inferencia en el worker sea consistente con el entrenamiento.

In [ ]:
import pickle

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# label_encoder.pkl → el worker lo carga para label_id → string
with open(os.path.join(MODEL_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(id2label, f)

print(f'Todo guardado en {MODEL_DIR}:')
for fname in sorted(os.listdir(MODEL_DIR)):
    kb = os.path.getsize(os.path.join(MODEL_DIR, fname)) // 1024
    print(f'  {fname:40s} {kb:,} KB')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.59it/s]

Todo guardado en ../model:
  config.json                              3 KB
  label_encoder.pkl                        1 KB
  model.safetensors                        261,780 KB
  tokenizer.json                           694 KB
  tokenizer_config.json                    0 KB
